<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
NeonCorp Operation
</font>
</h1>

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Loading The Dataset
</font>
</h2>


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path('data') if Path('data/train.csv').exists() else Path('.')
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
account = pd.read_csv(DATA_DIR / 'account.csv')
client = pd.read_csv(DATA_DIR / 'client.csv')
disp = pd.read_csv(DATA_DIR / 'disp.csv')
transactions = pd.read_csv(DATA_DIR / 'trans.csv', low_memory=False)
orders = pd.read_csv(DATA_DIR / 'order.csv')
cards = pd.read_csv(DATA_DIR / 'card.csv')
district = pd.read_csv(DATA_DIR / 'district.csv')

print('train:', train.shape, 'test:', test.shape)
train.head()

train: (477, 7) test: (205, 6)


,loan_id,account_id,date,amount,duration,payments,target
0,4973,67,1996-05-02,165960,24,6915,0
1,4986,97,1997-08-10,102876,12,8573,0
2,4988,103,1997-12-06,265320,36,7370,3
3,4990,110,1997-09-08,162576,36,4516,2
4,4996,132,1996-11-06,88440,12,7370,0


<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Feature Engineering
</font>
</h2>


In [2]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

# Cost matrix and scoring function
COST_MATRIX = np.array([
    [0, 3, 1, 2],
    [10, 0, 10, 4],
    [1, 3, 0, 2],
    [7, 4, 7, 0],
])

def competition_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    penalty = COST_MATRIX[y_true, y_pred].sum()
    max_penalty = COST_MATRIX[y_true].max(axis=1).sum()
    return 100 * max(0, 1 - penalty / max_penalty)

# ============================================================
# 1. PREPARE DATES
# ============================================================
train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])
account['date'] = pd.to_datetime(account['date'])
client['birth_date'] = pd.to_datetime(client['birth_date'])
transactions['date'] = pd.to_datetime(transactions['date'])

# Combine train+test for consistent feature engineering
all_loans = pd.concat([train, test], ignore_index=True)

# ============================================================
# 2. ACCOUNT FEATURES
# ============================================================
all_loans = all_loans.merge(account[['account_id','district_id','frequency','date']], on='account_id', how='left', suffixes=('','_acc'))
all_loans.rename(columns={'date_acc':'account_open_date'}, inplace=True)
all_loans['account_age_days'] = (all_loans['date'] - all_loans['account_open_date']).dt.days
all_loans['account_age_years'] = all_loans['account_age_days'] / 365.25

# Frequency encoding
freq_map = all_loans['frequency'].value_counts(normalize=True).to_dict()
all_loans['frequency_enc'] = all_loans['frequency'].map(freq_map)
all_loans = all_loans.drop(columns=['frequency'])

# ============================================================
# 3. CLIENT & DISPATCHER FEATURES (OWNER only)
# ============================================================
disp_owner = disp[disp['type'] == 'OWNER'][['account_id','client_id']].copy()
all_loans = all_loans.merge(disp_owner, on='account_id', how='left')
all_loans = all_loans.merge(client[['client_id','gender','birth_date','district_id']], on='client_id', how='left', suffixes=('_acc','_client'))
all_loans.rename(columns={'district_id_client':'client_district_id'}, inplace=True)
if 'district_id_acc' in all_loans.columns:
    all_loans.drop(columns=['district_id_acc'], inplace=True)

# Client age at loan time
all_loans['client_age_days'] = (all_loans['date'] - all_loans['birth_date']).dt.days
all_loans['client_age_years'] = all_loans['client_age_days'] / 365.25

# Gender encoding
all_loans['gender_enc'] = (all_loans['gender'] == 'M').astype(int)
all_loans.drop(columns=['gender','birth_date','client_id'], inplace=True)

# ============================================================
# 4. DISTRICT FEATURES
# ============================================================
all_loans = all_loans.merge(district, left_on='client_district_id', right_on='district_id', how='left', suffixes=('','_dist'))
all_loans.drop(columns=['district_id_dist'], errors='ignore', inplace=True)
all_loans.drop(columns=['A2','A3'], errors='ignore', inplace=True)

# ============================================================
# 5. TRANSACTION FEATURES (only before loan date)
# ============================================================
print('Building transaction features...')
trans_with_loan = transactions.merge(
    all_loans[['loan_id','account_id','date']].drop_duplicates(),
    on='account_id', how='inner'
)
# Only use transactions before the loan date
trans_before = trans_with_loan[trans_with_loan['date_x'] < trans_with_loan['date_y']].copy()
trans_before.rename(columns={'date_x':'trans_date','date_y':'loan_date'}, inplace=True)

def agg_trans(group):
    r = {}
    n = len(group)
    r['trans_count'] = n
    if n == 0:
        cols = ['trans_amount_mean','trans_amount_max','trans_amount_min','trans_amount_std',
                'trans_amount_median','trans_balance_mean','trans_balance_max','trans_balance_min',
                'trans_balance_std','trans_balance_last','trans_neg_balance_count',
                'trans_credit_count','trans_debit_count','trans_credit_ratio',
                'trans_days_span','trans_avg_daily_amount','trans_balance_trend',
                'trans_unique_k_symbol','trans_amount_range']
        for c in cols:
            r[c] = 0
        return pd.Series(r)
    r['trans_amount_mean'] = group['amount'].mean()
    r['trans_amount_max'] = group['amount'].max()
    r['trans_amount_min'] = group['amount'].min()
    r['trans_amount_std'] = group['amount'].std() if n > 1 else 0
    r['trans_amount_median'] = group['amount'].median()
    r['trans_amount_range'] = r['trans_amount_max'] - r['trans_amount_min']
    r['trans_balance_mean'] = group['balance'].mean()
    r['trans_balance_max'] = group['balance'].max()
    r['trans_balance_min'] = group['balance'].min()
    r['trans_balance_std'] = group['balance'].std() if n > 1 else 0
    r['trans_balance_last'] = group['balance'].iloc[-1]
    r['trans_neg_balance_count'] = (group['balance'] < 0).sum()
    r['trans_credit_count'] = (group['type'] == 'PRIJEM').sum()
    r['trans_debit_count'] = n - r['trans_credit_count']
    total = r['trans_credit_count'] + r['trans_debit_count']
    r['trans_credit_ratio'] = r['trans_credit_count'] / total if total > 0 else 0.5
    r['trans_days_span'] = (group['trans_date'].max() - group['trans_date'].min()).days
    r['trans_avg_daily_amount'] = r['trans_amount_mean'] / max(r['trans_days_span'], 1)
    r['trans_balance_trend'] = group['balance'].iloc[-1] - group['balance'].iloc[0]
    r['trans_unique_k_symbol'] = group['k_symbol'].nunique()
    return pd.Series(r)

trans_feats = trans_before.groupby('loan_id').apply(agg_trans).reset_index()
all_loans = all_loans.merge(trans_feats, on='loan_id', how='left')
print(f'Transaction features built for {len(trans_feats)} loans')

# ============================================================
# 6. ORDER FEATURES
# ============================================================
order_agg = orders.groupby('account_id').agg(
    order_count=('order_id', 'count'),
    order_amount_mean=('amount', 'mean'),
    order_amount_sum=('amount', 'sum'),
    order_amount_max=('amount', 'max'),
    order_amount_min=('amount', 'min'),
    order_k_symbol_nunique=('k_symbol', 'nunique'),
).reset_index()
all_loans = all_loans.merge(order_agg, on='account_id', how='left')

# ============================================================
# 7. CARD FEATURES
# ============================================================
card_with_disp = cards.merge(disp[['disp_id','account_id']], on='disp_id', how='left')
card_agg = card_with_disp.groupby('account_id').agg(
    card_count=('card_id', 'count'),
    has_gold=('type', lambda x: int((x == 'gold').any())),
    has_junior=('type', lambda x: int((x == 'junior').any())),
    has_classic=('type', lambda x: int((x == 'classic').any())),
).reset_index()
all_loans = all_loans.merge(card_agg, on='account_id', how='left')

# Fill NaN
for col in ['order_count','order_amount_mean','order_amount_sum','order_amount_max',
            'order_amount_min','order_k_symbol_nunique',
            'card_count','has_gold','has_junior','has_classic']:
    all_loans[col] = all_loans[col].fillna(0)

for col in trans_feats.columns:
    if col != 'loan_id':
        all_loans[col] = all_loans[col].fillna(0)

# ============================================================
# 8. LOAN-SPECIFIC DERIVED FEATURES
# ============================================================
all_loans['loan_amount'] = all_loans['amount']
all_loans['loan_duration'] = all_loans['duration']
all_loans['loan_payments'] = all_loans['payments']
all_loans['loan_to_balance'] = all_loans['amount'] / (all_loans['trans_balance_mean'].abs() + 1)
all_loans['monthly_payment_to_balance'] = all_loans['payments'] / (all_loans['trans_balance_mean'].abs() + 1)
all_loans['total_interest'] = all_loans['payments'] * all_loans['duration'] - all_loans['amount']
all_loans['interest_ratio'] = all_loans['total_interest'] / (all_loans['amount'] + 1)
all_loans['payment_per_amount'] = all_loans['payments'] / (all_loans['amount'] + 1)
all_loans['balance_to_loan'] = all_loans['trans_balance_mean'] / (all_loans['amount'] + 1)

# ============================================================
# 9. PREPARE FINAL FEATURES
# ============================================================
drop_cols = ['loan_id','account_id','date','target','client_district_id','account_open_date','district_id']
feature_cols = [c for c in all_loans.columns if c not in drop_cols]
print(f'Total features: {len(feature_cols)}')

train_data = all_loans[all_loans['loan_id'].isin(train['loan_id'])].copy()
test_data = all_loans[all_loans['loan_id'].isin(test['loan_id'])].copy()

X_train = train_data[feature_cols].values
y_train = train_data['target'].values.astype(int)
X_test = test_data[feature_cols].values

# Handle inf and nan
X_train = np.nan_to_num(X_train, nan=0.0, posinf=1e6, neginf=-1e6)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=1e6, neginf=-1e6)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

# ============================================================
# 10. MODEL TRAINING WITH COST-SENSITIVE LEARNING
# ============================================================
import xgboost as xgb
import lightgbm as lgb

# Cost matrix from the problem
COST = np.array([
    [0, 3, 1, 2],
    [10, 0, 10, 4],
    [1, 3, 0, 2],
    [7, 4, 7, 0],
])

# Class weights to handle imbalance
class_weights = {0: 1.0, 1: 5.0, 2: 0.5, 3: 5.0}
sample_weights = np.array([class_weights[y] for y in y_train])

def predict_cost_optimal(model, X):
    """Predict class that minimizes expected cost using cost matrix."""
    proba = model.predict_proba(X)
    expected_cost = proba @ COST
    return expected_cost.argmin(axis=1)

# XGBoost
print('\nTraining XGBoost...')
xgb_model = xgb.XGBClassifier(
    n_estimators=600, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0, gamma=0.1,
    objective='multi:softprob', num_class=4,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=42, n_jobs=-1,
)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
xgb_pred = predict_cost_optimal(xgb_model, X_test)
print(f'XGBoost preds: {np.bincount(xgb_pred, minlength=4)}')

# LightGBM
print('\nTraining LightGBM...')
lgb_model = lgb.LGBMClassifier(
    n_estimators=600, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0, num_leaves=31,
    objective='multiclass', num_class=4,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgb_model.fit(X_train, y_train, sample_weight=sample_weights)
lgb_pred = predict_cost_optimal(lgb_model, X_test)
print(f'LightGBM preds: {np.bincount(lgb_pred, minlength=4)}')

# Random Forest
print('\nTraining Random Forest...')
rf_model = RandomForestClassifier(
    n_estimators=600, max_depth=8, min_samples_leaf=3,
    class_weight='balanced', random_state=42, n_jobs=-1,
)
rf_model.fit(X_train, y_train)
rf_pred = predict_cost_optimal(rf_model, X_test)
print(f'RF preds: {np.bincount(rf_pred, minlength=4)}')

# Gradient Boosting
print('\nTraining Gradient Boosting...')
gb_model = GradientBoostingClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.03,
    subsample=0.8, min_samples_leaf=5, random_state=42,
)
gb_model.fit(X_train, y_train, sample_weight=sample_weights)
gb_pred = predict_cost_optimal(gb_model, X_test)
print(f'GB preds: {np.bincount(gb_pred, minlength=4)}')

# ============================================================
# 11. ENSEMBLE WITH PROBABILITY AVERAGING + COST-OPTIMAL
# ============================================================
print('\n--- Ensemble ---')
proba_xgb = xgb_model.predict_proba(X_test)
proba_lgb = lgb_model.predict_proba(X_test)
proba_rf = rf_model.predict_proba(X_test)
proba_gb = gb_model.predict_proba(X_test)

# Weighted average
weights = [0.35, 0.35, 0.15, 0.15]
avg_proba = weights[0]*proba_xgb + weights[1]*proba_lgb + weights[2]*proba_rf + weights[3]*proba_gb

expected_cost = avg_proba @ COST
test_pred = expected_cost.argmin(axis=1)

print(f'Final predictions: {np.bincount(test_pred, minlength=4)}')

# ============================================================
# 12. CROSS-VALIDATION
# ============================================================
print('\n--- Cross-Validation ---')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    w_tr = sample_weights[tr_idx]
    fold_model = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
        objective='multi:softprob', num_class=4, use_label_encoder=False,
        eval_metric='mlogloss', random_state=42,
    )
    fold_model.fit(X_tr, y_tr, sample_weight=w_tr)
    val_pred = predict_cost_optimal(fold_model, X_val)
    score = competition_score(y_val, val_pred)
    cv_scores.append(score)
    print(f'  Fold {fold+1}: score = {score:.2f}')
print(f'  Mean CV: {np.mean(cv_scores):.2f} (+/- {np.std(cv_scores):.2f})')

train_pred = predict_cost_optimal(xgb_model, X_train)
print(f'  Train score: {competition_score(y_train, train_pred):.2f}')
print('\n=== DONE! test_pred is ready. ===')

Building transaction features...


Transaction features built for 682 loans


Total features: 61
X_train: (477, 61), X_test: (205, 61)

Training XGBoost...


XGBoost preds: [ 64   6 118  17]

Training LightGBM...


LightGBM preds: [ 65   9 112  19]

Training Random Forest...


...

  Fold 1: score = 81.27


  Fold 2: score = 70.89


  Fold 3: score = 76.83


  Fold 4: score = 81.60


  Fold 5: score = 80.42
  Mean CV: 78.20 (+/- 4.03)
  Train score: 99.77

=== DONE! test_pred is ready. ===


<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
 Evaluation Metric</font>
</h2>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>

</font>
</p>


In [3]:
COST_MATRIX = np.array([
    [0, 3, 1, 2],
    [10, 0, 10, 4],
    [1, 3, 0, 2],
    [7, 4, 7, 0],
])

def competition_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    penalty = COST_MATRIX[y_true, y_pred].sum()
    max_penalty = COST_MATRIX[y_true].max(axis=1).sum()
    return 100 * max(0, 1 - penalty / max_penalty)

<h2 dir="ltr" align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>Result Generator Cell</b>
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
&nbsp;&nbsp;&nbsp;&nbsp;Run the cell below to generate the <code>result.zip</code> file. Please note that you must save the changes made in the notebook (<code>ctrl+s</code>) before running the cell below; otherwise, your score will be changed to zero at the end of the competition.
&nbsp;&nbsp;&nbsp;&nbsp;<br>
&nbsp;&nbsp;&nbsp;&nbsp;Also, if you are using Colab to run this notebook file, download the latest version of your notebook and place it inside the submission file before submitting the <code>result.zip</code> file.
</font>
</p>

In [4]:
import json
import zipfile

if test_pred is None:
    raise ValueError('test_pred is not set. Run the feature engineering cell first.')
submission = pd.DataFrame({
    'loan_id': test['loan_id'].astype(int),
    'target': np.asarray(test_pred, dtype=int),
})
if len(submission) != 205 or submission.loan_id.duplicated().any():
    raise ValueError('loan_id count or uniqueness is wrong.')
if not submission.target.isin([0, 1, 2, 3]).all():
    raise ValueError('Invalid target values.')
submission.to_csv('submission.csv', index=False)
with zipfile.ZipFile('result.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write('submission.csv', 'submission.csv')
    archive.write('notebook.ipynb', 'notebook.ipynb')
print('result.zip created.')

result.zip created.
